# Pasketti Phonetic — single-model inference walkthrough

End-to-end demo: load **one** trained checkpoint (NeMo Parakeet-TDT or WavLM-Large), run beam-search CTC / TDT decoding on a handful of audio files, and print the predicted IPA strings.

For the full 11-model ensemble + CatBoost reranker that produced the leaderboard score, run `make ensemble` instead — this notebook is only for sanity-checking a single training run.

In [ ]:
import os, sys
REPO = os.path.abspath('..')
sys.path.insert(0, os.path.join(REPO, 'src'))
sys.path.insert(0, os.path.join(REPO, 'src', '_compat'))
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')

In [ ]:
from gezi.common import *
from src import config
from src import dataset as ds
import gezi as gz, lele as le, importlib, torch

MODEL_DIR = os.path.join(REPO, 'working', 'online', '17', 'v17.fold0')
AUDIO_DIR = os.path.join(REPO, '..', 'input', 'pasketti', 'audio')
assert os.path.isdir(MODEL_DIR), f'Train a model first: make train-fold0 (looked under {MODEL_DIR})'

In [ ]:
# Restore the FLAGS used at training time.
config.init()        # set defaults
gz.restore_configs(MODEL_DIR)  # overlay flags.json
ic(FLAGS.model, FLAGS.backbone, FLAGS.fold)

In [ ]:
# Build the model and load the checkpoint.
Model = importlib.import_module(f'src.models.{FLAGS.model}').Model
model = Model().cuda().eval()
gz.load_weights(model, os.path.join(MODEL_DIR, 'model.pt'), strict=False)
print('model ready')

In [ ]:
# Run inference on the first few test clips.
import pandas as pd
fmt_csv = os.path.join(REPO, '..', 'input', 'pasketti', 'submission_format.csv')
df = pd.read_csv(fmt_csv).head(8)
df['audio_path'] = df['filename'].map(lambda f: os.path.join(AUDIO_DIR, f))
test_ds = ds.Dataset(df, mode='test')
dl = torch.utils.data.DataLoader(test_ds, batch_size=4, collate_fn=ds.collate_fn)

gz.set('do_generate', True)
preds = []
with torch.no_grad():
    for batch in dl:
        batch = {k: (v.cuda() if torch.is_tensor(v) else v) for k, v in batch.items()}
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            res = model(batch)
        preds.extend(res.get('pred_texts') or [])

for fn, p in zip(df['filename'], preds):
    print(f'{fn}: {p}')